In [2]:
import pandas as pd
import numpy as np

# =========================================================
# 1. GLOBAL SETTINGS
# =========================================================

SEED = 42
rng = np.random.default_rng(SEED)

N_TRANSACTIONS = 100_000
N_ORDERS = 65_000
N_STORES = 25

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

date_range = pd.date_range(START_DATE, END_DATE, freq="D")


# =========================================================
# 2. CREATE PRODUCT DIMENSION
# =========================================================

PRODUCT_STRUCTURE = {
    "Electronics": {
        "Accessories": 4,
        "Audio": 4,
        "Home Tech": 4
    },
    "Kitchenware": {
        "Cookware": 4,
        "Cutlery": 4,
        "Small Appliances": 4
    },
    "Furniture": {
        "Living Room": 2,
        "Bedroom": 2,
        "Office": 2
    }
}

products = []
product_counter = 1

for category, subcategories in PRODUCT_STRUCTURE.items():

    for subcategory, count in subcategories.items():

        for _ in range(count):

            product_id = f"P{product_counter:03d}"

            # ---------------------------------------------
            # Baseline 2024 economics
            # ---------------------------------------------

            if category == "Electronics":

                standard_unit_cost = rng.uniform(40, 180)
                baseline_margin = rng.uniform(0.30, 0.42)

            elif category == "Kitchenware":

                if subcategory == "Cookware":

                    standard_unit_cost = rng.uniform(20, 80)
                    baseline_margin = rng.uniform(0.44, 0.55)

                elif subcategory == "Cutlery":

                    standard_unit_cost = rng.uniform(10, 50)
                    baseline_margin = rng.uniform(0.46, 0.58)

                else:  # Small Appliances

                    standard_unit_cost = rng.uniform(35, 120)
                    baseline_margin = rng.uniform(0.28, 0.36)

            else:  # Furniture

                standard_unit_cost = rng.uniform(80, 250)
                baseline_margin = rng.uniform(0.32, 0.42)

            base_unit_price = (
                standard_unit_cost / (1 - baseline_margin)
            )

            products.append({
                "product_id": product_id,
                "product_category": category,
                "product_subcategory": subcategory,
                "standard_unit_cost_2024": round(
                    standard_unit_cost, 2
                ),
                "base_unit_price": round(
                    base_unit_price, 2
                )
            })

            product_counter += 1


dim_products = pd.DataFrame(products)

# Persistent differences in product popularity
dim_products["base_sales_weight"] = rng.dirichlet(
    np.ones(len(dim_products)) * 2.5
)


# =========================================================
# 3. CREATE STORE DIMENSION
# =========================================================

store_ids = [
    f"S{i:03d}"
    for i in range(1, N_STORES + 1)
]

dim_stores = pd.DataFrame({
    "store_id": store_ids
})

# Persistent differences in store activity
dim_stores["sales_weight"] = rng.dirichlet(
    np.ones(N_STORES) * 5
)

# Small persistent differences in discount behaviour
dim_stores["discount_tendency"] = rng.normal(
    loc=0,
    scale=0.006,
    size=N_STORES
)


# =========================================================
# 4. CREATE DAILY BUSINESS TRENDS
# =========================================================

daily_trends = pd.DataFrame({
    "date": date_range
})

daily_trends["year"] = (
    daily_trends["date"].dt.year
)

daily_trends["month"] = (
    daily_trends["date"].dt.month
)

daily_trends["quarter"] = (
    daily_trends["date"].dt.quarter
)

# ---------------------------------------------
# Sales volume
# ---------------------------------------------

daily_trends["volume_factor"] = np.where(
    daily_trends["year"] == 2024,
    1.00,
    1.15
)

# Mild seasonality
daily_trends["seasonality"] = (
    1
    + 0.06
    * np.sin(
        2 * np.pi
        * (daily_trends["month"] - 1)
        / 12
    )
)

daily_trends["transaction_weight"] = (
    daily_trends["volume_factor"]
    * daily_trends["seasonality"]
)


# ---------------------------------------------
# 2025 cost pressure
# ---------------------------------------------

daily_trends["cost_inflation"] = 0.0

mask_2025 = (
    daily_trends["year"] == 2025
)

progress_2025 = (
    daily_trends.loc[mask_2025, "date"]
    - pd.Timestamp("2025-01-01")
).dt.days / 364

daily_trends.loc[
    mask_2025,
    "cost_inflation"
] = (
    0.035
    + 0.065 * progress_2025
)


# ---------------------------------------------
# 2025 discount pressure
# ---------------------------------------------

daily_trends["discount_shift"] = 0.0

daily_trends.loc[
    mask_2025,
    "discount_shift"
] = (
    0.008
    + 0.018 * progress_2025
)


# =========================================================
# 5. GENERATE ORDERS
# =========================================================

date_probs = (
    daily_trends["transaction_weight"]
    / daily_trends["transaction_weight"].sum()
)

# Assign date and store at order level
order_dates = rng.choice(
    date_range,
    size=N_ORDERS,
    p=date_probs
)

order_stores = rng.choice(
    dim_stores["store_id"],
    size=N_ORDERS,
    p=dim_stores["sales_weight"]
)

dim_orders = pd.DataFrame({
    "order_id": [
        f"ORD{i:06d}"
        for i in range(1, N_ORDERS + 1)
    ],
    "transaction_date": pd.to_datetime(
        order_dates
    ),
    "store_id": order_stores
})

dim_orders = (
    dim_orders
    .sort_values("transaction_date")
    .reset_index(drop=True)
)


# =========================================================
# 6. ASSIGN 100,000 LINE ITEMS TO 65,000 ORDERS
# =========================================================
# Guarantee EVERY order has at least one line item.

base_assignments = (
    dim_orders["order_id"]
    .to_numpy()
)

remaining_line_items = (
    N_TRANSACTIONS - N_ORDERS
)

additional_assignments = rng.choice(
    dim_orders["order_id"],
    size=remaining_line_items,
    replace=True
)

order_assignments = np.concatenate([
    base_assignments,
    additional_assignments
])

# Shuffle so first 65k transactions do not map mechanically
# one-to-one to the first 65k orders
rng.shuffle(order_assignments)


fact_transactions = pd.DataFrame({
    "transaction_id": [
        f"TXN{i:06d}"
        for i in range(1, N_TRANSACTIONS + 1)
    ],
    "order_id": order_assignments
})

fact_transactions = fact_transactions.merge(
    dim_orders,
    on="order_id",
    how="left"
)

fact_transactions = (
    fact_transactions
    .sort_values("transaction_date")
    .reset_index(drop=True)
)

fact_transactions = fact_transactions.merge(
    daily_trends,
    left_on="transaction_date",
    right_on="date",
    how="left"
)


# =========================================================
# 7. ASSIGN PRODUCTS WITH 2025 MIX SHIFT
# =========================================================

product_ids = np.empty(
    N_TRANSACTIONS,
    dtype=object
)

base_weights = (
    dim_products["base_sales_weight"]
    .to_numpy()
    .copy()
)

subcategories = (
    dim_products["product_subcategory"]
    .to_numpy()
)

small_appliance_mask = (
    subcategories == "Small Appliances"
)

for day_date in date_range:

    row_mask = (
        fact_transactions["transaction_date"]
        == day_date
    )

    n_day_txns = row_mask.sum()

    if n_day_txns == 0:
        continue

    weights = base_weights.copy()

    # ---------------------------------------------
    # 2025 mix shift toward lower-margin
    # Small Appliances
    # ---------------------------------------------

    if day_date.year == 2025:

        year_progress = (
            day_date
            - pd.Timestamp("2025-01-01")
        ).days / 364

        weights[
            small_appliance_mask
        ] *= (
            1
            + 0.65 * year_progress
        )

    weights = (
        weights / weights.sum()
    )

    product_ids[row_mask] = rng.choice(
        dim_products["product_id"],
        size=n_day_txns,
        p=weights
    )


fact_transactions["product_id"] = (
    product_ids
)


# =========================================================
# 8. ASSIGN LINE-ITEM QUANTITY
# =========================================================
# Quantity = units of this particular product
# within the order line.

fact_transactions["quantity"] = rng.choice(
    [1, 2, 3, 4, 5, 6],
    size=N_TRANSACTIONS,
    p=[
        0.42,
        0.28,
        0.15,
        0.08,
        0.05,
        0.02
    ]
)


# =========================================================
# 9. MERGE PRODUCT + STORE ECONOMICS
# =========================================================

fact_transactions = fact_transactions.merge(
    dim_products[
        [
            "product_id",
            "product_category",
            "product_subcategory",
            "standard_unit_cost_2024",
            "base_unit_price"
        ]
    ],
    on="product_id",
    how="left"
)

fact_transactions = fact_transactions.merge(
    dim_stores[
        [
            "store_id",
            "discount_tendency"
        ]
    ],
    on="store_id",
    how="left"
)


# =========================================================
# 10. GENERATE UNIT COST
# =========================================================
# unit_cost = fully landed inventory cost

category_cost_multiplier = np.select(
    [
        fact_transactions["product_category"]
        == "Electronics",

        fact_transactions["product_category"]
        == "Kitchenware",

        fact_transactions["product_category"]
        == "Furniture"
    ],
    [
        1.35,
        1.00,
        0.70
    ],
    default=1.00
)

cost_noise = rng.normal(
    0,
    0.018,
    N_TRANSACTIONS
)

fact_transactions["unit_cost"] = (
    fact_transactions[
        "standard_unit_cost_2024"
    ]
    * (
        1
        + (
            fact_transactions["cost_inflation"]
            * category_cost_multiplier
        )
        + cost_noise
    )
)

fact_transactions["unit_cost"] = (
    fact_transactions["unit_cost"]
    .clip(lower=1)
    .round(2)
)


# =========================================================
# 11. GENERATE DISCOUNT RATE
# =========================================================

base_discount = (
    rng.beta(
        2,
        10,
        N_TRANSACTIONS
    )
    * 0.18
)

fact_transactions["discount_rate"] = (
    base_discount
    + fact_transactions["discount_shift"]
    + fact_transactions["discount_tendency"]
)

fact_transactions["discount_rate"] = (
    fact_transactions["discount_rate"]
    .clip(0, 0.22)
    .round(4)
)


# =========================================================
# 12. GENERATE UNIT PRICE
# =========================================================
# 2025 price growth exists but does not fully offset
# cost and discount pressure.

fact_transactions["price_growth"] = np.where(
    fact_transactions["year"] == 2024,
    0.00,
    0.025
)

price_noise = rng.normal(
    0,
    0.012,
    N_TRANSACTIONS
)

fact_transactions["unit_price"] = (
    fact_transactions["base_unit_price"]
    * (
        1
        + fact_transactions["price_growth"]
        + price_noise
    )
)

fact_transactions["unit_price"] = (
    fact_transactions["unit_price"]
    .clip(lower=1)
    .round(2)
)


# =========================================================
# 13. FINAL FACT TABLE
# =========================================================

fact_transactions = fact_transactions[
    [
        "transaction_id",
        "order_id",
        "transaction_date",
        "store_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount_rate",
        "unit_cost"
    ]
]


# =========================================================
# 14. VALIDATION DATASET
# =========================================================

validation = fact_transactions.merge(
    dim_products[
        [
            "product_id",
            "product_category",
            "product_subcategory"
        ]
    ],
    on="product_id",
    how="left"
)

validation["sales_revenue"] = (
    validation["quantity"]
    * validation["unit_price"]
    * (
        1
        - validation["discount_rate"]
    )
)

validation["cogs"] = (
    validation["quantity"]
    * validation["unit_cost"]
)

validation["gross_profit"] = (
    validation["sales_revenue"]
    - validation["cogs"]
)

validation["year"] = (
    validation[
        "transaction_date"
    ].dt.year
)

validation["quarter"] = (
    validation[
        "transaction_date"
    ].dt.quarter
)


# =========================================================
# 15. YEARLY FINANCIAL SUMMARY
# =========================================================

yearly_summary = (
    validation
    .groupby("year")
    .agg(
        revenue=(
            "sales_revenue",
            "sum"
        ),
        cogs=(
            "cogs",
            "sum"
        ),
        gross_profit=(
            "gross_profit",
            "sum"
        ),
        orders=(
            "order_id",
            "nunique"
        ),
        line_items=(
            "transaction_id",
            "count"
        ),
        units=(
            "quantity",
            "sum"
        )
    )
    .reset_index()
)

yearly_summary[
    "gross_margin_pct"
] = (
    yearly_summary[
        "gross_profit"
    ]
    / yearly_summary[
        "revenue"
    ]
    * 100
)

print("\nYEARLY FINANCIAL SUMMARY")
print(
    yearly_summary.round(2)
)


# =========================================================
# 16. QUARTERLY FINANCIAL SUMMARY
# =========================================================

quarterly_summary = (
    validation
    .groupby(
        [
            "year",
            "quarter"
        ]
    )
    .agg(
        revenue=(
            "sales_revenue",
            "sum"
        ),
        cogs=(
            "cogs",
            "sum"
        ),
        gross_profit=(
            "gross_profit",
            "sum"
        ),
        orders=(
            "order_id",
            "nunique"
        ),
        line_items=(
            "transaction_id",
            "count"
        ),
        units=(
            "quantity",
            "sum"
        )
    )
    .reset_index()
)

quarterly_summary[
    "gross_margin_pct"
] = (
    quarterly_summary[
        "gross_profit"
    ]
    / quarterly_summary[
        "revenue"
    ]
    * 100
)

print("\nQUARTERLY FINANCIAL SUMMARY")
print(
    quarterly_summary.round(2)
)


# =========================================================
# 17. BASKET VALIDATION
# =========================================================
# Confirms that we genuinely have 65,000 orders
# and 100,000 transaction lines.

basket_summary = (
    fact_transactions
    .groupby("order_id")
    .agg(
        line_items=(
            "transaction_id",
            "count"
        ),
        basket_units=(
            "quantity",
            "sum"
        )
    )
    .reset_index()
)

print("\nBASKET VALIDATION")
print(
    "Unique orders:",
    fact_transactions[
        "order_id"
    ].nunique()
)

print(
    "Transaction lines:",
    len(fact_transactions)
)

print(
    "\nLine items per order:"
)

print(
    basket_summary[
        "line_items"
    ].describe().round(2)
)

print(
    "\nUnits per order:"
)

print(
    basket_summary[
        "basket_units"
    ].describe().round(2)
)


# =========================================================
# 18. OPTIONAL: SAVE CSV FILES
# =========================================================

fact_transactions.to_csv(
    "fact_transactions.csv",
    index=False
)

dim_products[
    [
        "product_id",
        "product_category",
        "product_subcategory"
    ]
].to_csv(
    "dim_products.csv",
    index=False
)

dim_stores[
    [
        "store_id"
    ]
].to_csv(
    "dim_stores.csv",
    index=False
)


YEARLY FINANCIAL SUMMARY
   year      revenue         cogs  gross_profit  orders  line_items   units  \
0  2024  16522814.20  10518394.34    6004419.86   30129       46418   98054   
1  2025  18997054.93  12890849.21    6106205.72   34871       53582  113349   

   gross_margin_pct  
0             36.34  
1             32.14  

QUARTERLY FINANCIAL SUMMARY
   year  quarter     revenue        cogs  gross_profit  orders  line_items  \
0  2024        1  4142956.73  2638375.73    1504581.00    7694       11774   
1  2024        2  4391844.98  2794115.89    1597729.09    7864       12149   
2  2024        3  4060919.97  2585715.64    1475204.33    7382       11411   
3  2024        4  3927092.51  2500187.08    1426905.43    7189       11084   
4  2025        1  4844440.45  3176949.34    1667491.11    8755       13441   
5  2025        2  5035724.36  3385827.33    1649897.03    9188       14201   
6  2025        3  4619774.78  3173113.13    1446661.65    8619       13184   
7  2025        4 